In [1]:
import gradio as gr
import numpy as np
import tensorflow as tf
import gradio as gr
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_i


In [2]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image

# Define the custom layer
class FixedDropout(tf.keras.layers.Dropout):
    def _get_noise_shape(self, inputs):
        if self.noise_shape is None:
            return self.noise_shape
        symbolic_shape = tf.shape(inputs)
        noise_shape = [symbolic_shape[axis] if shape is None else shape for axis, shape in enumerate(self.noise_shape)]
        return tuple(noise_shape)

# Load all three models with custom object scope
with tf.keras.utils.custom_object_scope({'FixedDropout': FixedDropout}):
    modelb4 = load_model('efficientnetb4.h5')
    modelb5 = load_model('efficientnetb5.h5')
    model_resnet152v2 = load_model('ResNet152V2.h5')

# Define a function to preprocess the image and make predictions
def predict_image(img):
    # Convert the Gradio image object to a PIL Image
    img_pil = Image.fromarray(img.astype('uint8'), 'RGB')
    # Resize the image to the input size expected by the models
    img_resized = img_pil.resize((380, 380))  # Resize to match the model's input size
    # Convert the PIL image to a numpy array
    img_array = keras_image.img_to_array(img_resized)
    # Preprocess the image (e.g., normalization)
    img_array = img_array / 255.0  # Normalize pixel values
    # Expand the dimensions to match the input shape expected by the models
    img_array = np.expand_dims(img_array, axis=0)
    
  # Make predictions with all three models
    predictions_b4 = modelb4.predict(img_array)
    predictions_b5 = modelb5.predict(img_array)
    predictions_resnet152v2 = model_resnet152v2.predict(img_array)

    # Take the maximum value of predictions from all three models
    ensemble_prediction = np.maximum(np.maximum(predictions_b4, predictions_b5), predictions_resnet152v2)

    # Get the index of the maximum prediction
    predicted_class_index = np.argmax(ensemble_prediction)
    # Example: If your model has a list of class labels, you can return the predicted class label
    class_labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferate']  # Replace with your actual class labels
    predicted_class_label = class_labels[predicted_class_index]
    # Return the predicted class label
    return predicted_class_label

# Create a Gradio interface
iface = gr.Interface(predict_image, 
                     gr.inputs.Image(shape=(380, 380)),  # Update shape to match the input size of the models
                     gr.outputs.Label(),
                     title="Diabetic Retinopathy Detection",
                     description="The most probable label will be predicted by the model when you upload an image.",
                     theme="soft")
# Launch the interface
iface.launch(share=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12044\2214733212.py:54: GradioDeprecationWarning: Usage of gradio.inputs is deprecated, and will not be supported in the future, please import your component from gradio.components
  gr.inputs.Image(shape=(380, 380)),  # Update shape to match the input size of the models
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12044\2214733212.py:54: GradioDeprecationWarning: `optional` parameter is deprecated, and it has no effect
  gr.inputs.Image(shape=(380, 380)),  # Update shape to match the input size of the models
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12044\2214733212.py:55: GradioDeprecationWarning: Usage of gradio.outputs is deprecated, and will not be supported in the future, please import your components from gradio.components
  gr.outputs.Label(),
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12044\2214733212.py:55: GradioUnusedKwargWarning: You have unused kwarg parameters in Label, please remove them: {'type': 'auto'}
  gr.outputs.La

Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.50.0, however version 4.44.1 is available, please upgrade.
--------

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


1/1 [==============================] - 10s 10s/step
